# Agente Noticias - Win Internet

Este notebook recorre, celda por celda y al estilo del curso `lca-lc-foundations`, el agente que:

1. Busca noticias de IA en Tavily con varias queries especializadas.
2. Las evalua con un LLM y salida estructurada (Pydantic).
3. Resume y genera un correo HTML.
4. Pausa para que apruebes (**Human-in-the-loop** con `interrupt`).
5. Envia el correo via Outlook Desktop (`pywin32`).

Todo queda trazado en **LangSmith** bajo el proyecto `agente-noticias-win`.

## 1. Setup

Importamos el paquete `agente_noticias`, que al cargarse lee el `.env` del curso y activa el tracing de LangSmith.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import agente_noticias  # carga el .env y activa LangSmith tracing
import os
print('LANGSMITH_TRACING =', os.getenv('LANGSMITH_TRACING'))
print('LANGSMITH_PROJECT =', os.getenv('LANGSMITH_PROJECT'))
print('Destinatario      =', agente_noticias.config.get_recipient())

## 1.5. Visualizar la estructura del grafo

Antes de ejecutar nada, podemos ver el mapa de nodos y aristas con `draw_mermaid_png()`.

In [ ]:
from agente_noticias.graph import build_graph
from IPython.display import Image, display

graph = build_graph()

# Diagrama PNG (usa la API publica de mermaid.ink)
display(Image(graph.get_graph().draw_mermaid_png()))

# Texto mermaid (por si la API esta caida, copia y pega en mermaid.live)
print(graph.get_graph().draw_mermaid())

## 2. Nodo 1: Researcher (Tavily)

Como en `notebooks/module-1/1.2_web_search.ipynb` del curso, usamos Tavily como buscador. Aqui lanzamos varias queries en paralelo (telecom, Peru, competidores, regulador, atencion al cliente) y deduplicamos por URL.

In [ ]:
from agente_noticias.nodes.researcher import researcher_node
from agente_noticias.config import SEARCH_QUERIES

print(f'Queries configuradas: {len(SEARCH_QUERIES)}')
for q in SEARCH_QUERIES:
    print(' -', q['query'])

research = researcher_node({})
raw_articles = research['raw_articles']
print(f'\nArticulos unicos: {len(raw_articles)}')
for a in raw_articles[:5]:
    print(f'  [{a.score:.2f}] {a.title[:80]}')

## 3. Nodo 2: Evaluator (LLM + Pydantic)

Cada articulo pasa por `gpt-5-nano` con `with_structured_output(ArticleEvaluation)`. El LLM puntua de 0 a 10 la relevancia para Win Internet y asigna areas de impacto.

Esto sigue el patron del curso (modulo 1, structured outputs) pero aplicado a clasificacion de noticias.

In [ ]:
from agente_noticias.nodes.evaluator import evaluator_node

eval_out = evaluator_node({'raw_articles': raw_articles})
selected = eval_out['selected_articles']
print(f'Seleccionados: {len(selected)} de {len(eval_out["scored_articles"])}\n')
for item in selected:
    e = item.evaluation
    print(f'[{e.relevance_score}/10] {item.article.title[:70]}')
    print(f'    Areas: {e.impact_areas}')
    print(f'    Por que: {e.why_matters_for_win}\n')

## 4. Nodo 3: Summarizer

Consolida un `headline` y 3 bullets TL;DR. Tambien con structured output (`Briefing`).

In [ ]:
from agente_noticias.nodes.summarizer import summarizer_node

sum_out = summarizer_node({'selected_articles': selected})
briefing = sum_out['briefing']
print('Headline:', briefing.headline)
print('\nTL;DR:')
for b in briefing.tldr:
    print(' -', b)

## 5. Nodo 4: Email Writer

Renderiza la plantilla HTML compatible con Outlook (tablas + CSS inline) y guarda `output/preview.html`.

In [ ]:
from agente_noticias.nodes.email_writer import email_writer_node
from IPython.display import HTML, display

ew_out = email_writer_node({
    'briefing': briefing,
    'selected_articles': selected,
})
print('Subject:     ', ew_out['subject'])
print('Preview path:', ew_out['preview_path'])
print('Run id:      ', ew_out['run_id'])

display(HTML(ew_out['html_body']))

## 6. Grafo completo + Human-in-the-loop

Ahora ensamblamos el grafo en LangGraph (igual al patron de `notebooks/module-3/3.5_email_agent.ipynb`). El nodo `approval_gate` pausa con `interrupt()` antes de enviar.

In [ ]:
from agente_noticias.graph import build_graph, default_config

graph = build_graph()
config = default_config(thread_id='nb-demo-1')

result = graph.invoke({}, config)
interrupts = result.get('__interrupt__')
print('Interrupts:', interrupts)
print('\nPayload visible para el humano:')
if interrupts:
    for k, v in interrupts[0].value.items():
        print(f'  {k}: {v}')

### Aprobar y enviar

Para enviar el correo a `jquispeh@win.pe` (requiere Outlook abierto), reanuda con `Command(resume=True)`.

In [ ]:
from langgraph.types import Command

final = graph.invoke(Command(resume=True), config)
print('approved:    ', final.get('approved'))
print('send_result: ', final.get('send_result'))
print('run_id:      ', final.get('run_id'))

### Rechazar (alternativa)

Si en lugar de aprobar quieres rechazar, reinicia con otro `thread_id` y pasa `Command(resume=False)`.

In [ ]:
config_reject = default_config(thread_id='nb-demo-2')
_ = graph.invoke({}, config_reject)
rejected = graph.invoke(Command(resume=False), config_reject)
print('approved:    ', rejected.get('approved'))
print('send_result: ', rejected.get('send_result'))

## 7. Trazabilidad LangSmith

Abre https://smith.langchain.com y busca el proyecto **`agente-noticias-win`**.
Cada nodo (`researcher`, `evaluator`, `summarizer`, `email_writer`, `approval_gate`, `outlook_sender`) aparece como un span. El `run_id` que se incluye en el footer del correo permite saltar directo al run desde Outlook.